# Treino RGB area+energia (Deep Watershed) no KAGGLE

Igual ao do Colab, mas adaptado pro Kaggle (sem Google Drive).

## Passos
1. Crie um **Dataset** no Kaggle com o `rgb_dataset_big.zip` (menu **+ Create → New Dataset**, ou no notebook **Add Data → Upload**).
2. Neste notebook: painel direito → **Add Data** → selecione esse dataset. Ele aparece em `/kaggle/input/...`.
3. Painel direito → **Settings**: **Accelerator = GPU T4 x2** (ou P100) e **Internet = On** (precisa para baixar os pesos ImageNet).
4. Rode tudo. O melhor checkpoint fica em `/kaggle/working/out/` (baixavel no fim, aba **Output**).

## Resumir (continuar de onde parou)
Como o `/kaggle/working` zera a cada sessao: ao terminar uma sessao, salve o output como Dataset
(**Output → New Dataset**) e, na proxima sessao, **Add Data** desse dataset com o `last_energy_model.pt`.
O notebook detecta e continua automaticamente.


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"  # anti-fragmentacao CUDA (evita OOM)
# 1) Dependencias (torch ja vem no Colab)
!pip -q install segmentation-models-pytorch==0.5.0 rasterio shapely tqdm
import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| bf16:", torch.cuda.is_bf16_supported())


In [ ]:
# 2) Dados no Kaggle (sem Drive)
import os, glob, zipfile
def _find_root(base):
    h=glob.glob(base+"/**/images/train", recursive=True)
    return os.path.dirname(os.path.dirname(h[0])) if h else None

# 1) dataset ja extraido em /kaggle/input ?
DATASET_DIR=_find_root("/kaggle/input")
# 2) senao, procura um .zip no input e extrai pra /kaggle/working
if DATASET_DIR is None:
    zips=glob.glob("/kaggle/input/**/*.zip", recursive=True)
    assert zips, "Adicione o dataset (Add Data) com a pasta images/masks ou o .zip."
    os.makedirs("/kaggle/working/data", exist_ok=True)
    with zipfile.ZipFile(zips[0]) as z: z.extractall("/kaggle/working/data")
    DATASET_DIR=_find_root("/kaggle/working/data")
OUT_DIR="/kaggle/working/out"; os.makedirs(OUT_DIR, exist_ok=True)
print("DATASET_DIR =", DATASET_DIR)
print("train:", len(glob.glob(DATASET_DIR+"/images/train/*.tif")), "| val:", len(glob.glob(DATASET_DIR+"/images/val/*.tif")))

# resume: se voce adicionou um dataset com last_energy_model.pt, ele e detectado
RESUME_CKPT=(glob.glob("/kaggle/input/**/last_energy_model.pt", recursive=True) or [None])[0]
print("checkpoint para resume:", RESUME_CKPT)


In [ ]:
# 3) Treinador area+energia (mesmo do script 56) com AMP automatico
import os, glob, random, math, json
import numpy as np, rasterio, torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

CFG = dict(
    arch="unetplusplus",   # ou "unet" (mais leve)
    encoder="resnet34",    # resnet34 cabe na T4; "resnet50" se sobrar memoria
    image_size=512,
    batch_size=8,          # se faltar memoria: 6 -> 4
    epochs=30,
    lr=2e-4,
    wd=1e-4,
    energy_weight=1.5,
    num_workers=4,
    seed=2026,
)

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if bf16 else torch.float16
use_scaler = not bf16
print("precisao:", "bf16" if bf16 else "fp16+scaler")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def build_pairs(root, split):
    pairs=[]
    for img in sorted(glob.glob(f"{root}/images/{split}/*.tif")):
        m = img.replace("/images/","/masks/").replace("_ortho_","_ortholabel_")
        if os.path.exists(m): pairs.append((img, m))
    return pairs

def ratio_class(r):
    if r==0: return "empty"
    if r<0.05: return "small"
    if r<0.25: return "medium"
    if r<0.75: return "large"
    return "almost_full"

class EnergyDataset(Dataset):
    def __init__(self, pairs, image_size=512, augment=False, scale_jitter=True):
        self.pairs=pairs; self.image_size=image_size; self.augment=augment; self.scale_jitter=scale_jitter
    def __len__(self): return len(self.pairs)
    def _read(self, ip, mp):
        with rasterio.open(ip) as s: rgb=s.read([1,2,3]).astype(np.float32)
        if rgb.max()>2: rgb/=255.0
        rgb=np.clip(rgb,0,1)
        with rasterio.open(mp) as s:
            area=(s.read(1)>0).astype(np.float32); energy=s.read(2).astype(np.float32)/255.0
        return rgb, np.stack([area,energy],0).astype(np.float32)
    def _photometric(self, x):
        if random.random()<0.5: x=x+random.uniform(-0.18,0.18)
        if random.random()<0.5: x=(x-0.5)*random.uniform(0.7,1.3)+0.5
        if random.random()<0.5: x=torch.clamp(x,0,1)**random.uniform(0.7,1.4)
        if random.random()<0.7:
            g=x.mean(0,keepdim=True); s=random.uniform(0.1,1.0); x=s*x+(1-s)*g  # dessatura: forca textura/forma
        if random.random()<0.5: x=x*torch.tensor([random.uniform(0.8,1.2) for _ in range(3)]).view(3,1,1)
        if random.random()<0.2: x=x.mean(0,keepdim=True).repeat(3,1,1)  # cinza total
        if random.random()<0.3: x=x+torch.randn_like(x)*0.025
        return torch.clamp(x,0,1)
    def _scale_jitter(self, x, y):
        s=random.uniform(0.75,1.3); H=self.image_size; nh=max(8,int(round(H*s)))
        x=F.interpolate(x[None],size=(nh,nh),mode="bilinear",align_corners=False)[0]
        ya=F.interpolate(y[0:1][None],size=(nh,nh),mode="nearest")[0]
        ye=F.interpolate(y[1:2][None],size=(nh,nh),mode="bilinear",align_corners=False)[0]
        y=torch.cat([ya,ye],0)
        if nh>=H:
            t=random.randint(0,nh-H); l=random.randint(0,nh-H)
            return x[:,t:t+H,l:l+H], y[:,t:t+H,l:l+H]
        pad=H-nh; t=random.randint(0,pad); l=random.randint(0,pad)
        return F.pad(x,(l,pad-l,t,pad-t),mode="reflect"), F.pad(y,(l,pad-l,t,pad-t),value=0)
    def __getitem__(self, i):
        ip,mp=self.pairs[i]; rgb,tgt=self._read(ip,mp)
        x=torch.from_numpy(rgb); y=torch.from_numpy(tgt); H=self.image_size
        if x.shape[-2:]!=(H,H):
            x=F.interpolate(x[None],size=(H,H),mode="bilinear",align_corners=False)[0]
            ya=F.interpolate(y[0:1][None],size=(H,H),mode="nearest")[0]
            ye=F.interpolate(y[1:2][None],size=(H,H),mode="bilinear",align_corners=False)[0]
            y=torch.cat([ya,ye],0)
        if self.augment:
            if self.scale_jitter and random.random()<0.5: x,y=self._scale_jitter(x,y)
            if random.random()<0.5: x=torch.flip(x,[2]); y=torch.flip(y,[2])
            if random.random()<0.5: x=torch.flip(x,[1]); y=torch.flip(y,[1])
            k=random.randint(0,3)
            if k: x=torch.rot90(x,k,[1,2]); y=torch.rot90(y,k,[1,2])
            x=self._photometric(x)
        mean=torch.tensor([0.485,0.456,0.406]).view(3,1,1); std=torch.tensor([0.229,0.224,0.225]).view(3,1,1)
        return {"image":((x-mean)/std).float(), "target":y.float()}

def tversky(logits,t,a=0.35,b=0.65,eps=1e-7):
    p=torch.sigmoid(logits).flatten(1); y=t.flatten(1)
    tp=(p*y).sum(1); fp=(p*(1-y)).sum(1); fn=((1-p)*y).sum(1)
    return (1-(tp+eps)/(tp+a*fp+b*fn+eps)).mean()

def loss_fn(logits,target,ew):
    al,el=logits[:,0:1],logits[:,1:2]; at,et=target[:,0:1],target[:,1:2]
    area=F.binary_cross_entropy_with_logits(al,at)+tversky(al,at)
    energy=F.l1_loss(torch.sigmoid(el),et)
    return area+ew*energy

def area_dice(logits,target,thr=0.5,eps=1e-7):
    p=(torch.sigmoid(logits[:,0:1])>=thr).float().flatten(1); y=target[:,0:1].flatten(1)
    tp=(p*y).sum(1); d=(2*tp+eps)/(2*tp+(p*(1-y)).sum(1)+((1-p)*y).sum(1)+eps); return float(d.mean())

def energy_mae(logits,target): return float(F.l1_loss(torch.sigmoid(logits[:,1:2]),target[:,1:2]))
print("trainer pronto.")


In [ ]:
# 4) Treinar
set_seed(CFG["seed"])
tr=build_pairs(DATASET_DIR,"train"); va=build_pairs(DATASET_DIR,"val")
print("train",len(tr),"val",len(va))

cls=[]
for _,mp in tr:
    with rasterio.open(mp) as s: cls.append(ratio_class(float((s.read(1)>0).mean())))
cw={"empty":0.6,"small":3.0,"medium":2.5,"large":1.0,"almost_full":0.5}
weights=torch.tensor([cw.get(c,1.0) for c in cls],dtype=torch.double)
sampler=WeightedRandomSampler(weights,len(weights),replacement=True)

tl=DataLoader(EnergyDataset(tr,CFG["image_size"],True,True),batch_size=CFG["batch_size"],
              sampler=sampler,num_workers=CFG["num_workers"],pin_memory=True,drop_last=True)
vl=DataLoader(EnergyDataset(va,CFG["image_size"],False),batch_size=CFG["batch_size"],shuffle=False,
              num_workers=CFG["num_workers"],pin_memory=True)

model=smp.create_model(arch={"unet":"Unet","unetplusplus":"UnetPlusPlus"}[CFG["arch"]],
        encoder_name=CFG["encoder"],encoder_weights="imagenet",in_channels=3,classes=2,activation=None).to(dev)
opt=torch.optim.AdamW(model.parameters(),lr=CFG["lr"],weight_decay=CFG["wd"])
sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=CFG["epochs"],eta_min=CFG["lr"]*0.05)
scaler=torch.amp.GradScaler('cuda', enabled=use_scaler)
cfg_save={"task":"rgb_area_energy","architecture":CFG["arch"],"encoder_name":CFG["encoder"],
          "in_channels":3,"classes":2,"output_channels":{"0":"area","1":"energy"}}

RESUME=True  # continua do ultimo checkpoint no Drive, se existir (nao perde epocas)
import os
best=-1; hist=[]; start_ep=1
_ck=RESUME_CKPT if (RESUME_CKPT and os.path.exists(RESUME_CKPT)) else f"{OUT_DIR}/last_energy_model.pt"
if RESUME and os.path.exists(_ck):
    _s=torch.load(_ck,map_location=dev)
    model.load_state_dict(_s["model_state_dict"])
    if "optimizer_state_dict" in _s: opt.load_state_dict(_s["optimizer_state_dict"])
    if "best" in _s: best=_s["best"]
    start_ep=int(_s.get("epoch",0))+1
    for _ in range(start_ep-1): sched.step()   # alinha o agendador de LR
    print(f">>> RESUMINDO da epoca {start_ep} (melhor score ate agora={best:.3f})")
for ep in range(start_ep,CFG["epochs"]+1):
    model.train(); tot=n=0
    for b in tqdm(tl,desc=f"train {ep}",leave=False):
        x=b["image"].to(dev,non_blocking=True); y=b["target"].to(dev,non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.autocast("cuda",dtype=amp_dtype):
            logits=model(x); loss=loss_fn(logits,y,CFG["energy_weight"])
        if use_scaler:
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update()
        else:
            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()
        tot+=float(loss); n+=1
    sched.step()
    model.eval(); vd=vm=vn=0
    with torch.no_grad():
        for b in tqdm(vl,desc=f"val {ep}",leave=False):
            x=b["image"].to(dev); y=b["target"].to(dev)
            with torch.autocast("cuda",dtype=amp_dtype):
                logits=model(x)
            vd+=area_dice(logits,y); vm+=energy_mae(logits,y); vn+=1
    vd/=vn; vm/=vn; score=vd+max(0.0,1.0-2.0*vm)
    hist.append({"epoch":ep,"train_loss":tot/n,"val_area_dice":vd,"val_energy_mae":vm,"score":score})
    import pandas as pd; pd.DataFrame(hist).to_csv(f"{OUT_DIR}/history.csv",index=False)
    print(f"ep {ep:02d}/{CFG['epochs']} loss={tot/n:.3f} val_area_dice={vd:.3f} energy_mae={vm:.3f} score={score:.3f}")
    torch.save({"epoch":ep,"model_state_dict":model.state_dict(),"optimizer_state_dict":opt.state_dict(),"config":cfg_save,"best":best},f"{OUT_DIR}/last_energy_model.pt")
    if score>best:
        best=score
        torch.save({"epoch":ep,"model_state_dict":model.state_dict(),"optimizer_state_dict":opt.state_dict(),"config":cfg_save,"score":best,"best":best},
                   f"{OUT_DIR}/best_energy_model.pt")
        print(f"  novo melhor: {best:.3f} (ep {ep})")
print("FIM. melhor score=",best,"checkpoint em",OUT_DIR)


## Baixar o modelo
No fim, aba **Output** (painel direito) → baixe `out/best_energy_model.pt`.
Coloque em `C:\tcc_sentinel2\outputs\rgb_fields\energy_colab_big\` no PC e rode a avaliacao local:

```bash
python scripts/55_object_level_metrics.py --checkpoint outputs/rgb_fields/energy_colab_big/best_energy_model.pt \
  --root data/prepared/rgb_fields_yuri_v3e --split val --energy --tta --min-seed-dist 70 --energy-seed-thr 0.4 --min-area-px 120
```
